In [ ]:
# Installs the arclog wheel from the dataset, offline, then runs the concept
# case program of aba27056 on its task, whose test output the dataset holds out.
import glob, subprocess, sys

wheels = glob.glob("/kaggle/input/**/arclog-*.whl", recursive=True)
print("wheels:", wheels)
assert len(wheels) == 1, wheels
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", wheels[0]], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
assert r.returncode == 0

In [ ]:
import json, time
import arclog

TASK_ID = "aba27056"
[task_path] = glob.glob(f"/kaggle/input/**/{TASK_ID}.json", recursive=True)
[stream_path] = glob.glob(f"/kaggle/input/**/{TASK_ID}.stream.json", recursive=True)
with open(task_path) as f:
    task_text = f.read()
with open(stream_path) as f:
    stream = json.load(f)
assert all("output" not in pair for pair in json.loads(task_text)["test"]), "the task file holds a test output"

task = arclog.Task.from_json(task_text, arclog.World.Open)
session = arclog.Session(task)
print("task:", task.train_count, "train pairs,", task.test_count, "test inputs;", "stream:", len(stream), "tokens")

In [ ]:
def mask_admits(mask, token_id):
    return bool(mask[token_id // 8] >> (token_id % 8) & 1)


def decode(session, stream):
    """Steps a decoder through `stream`: each model token must be admissible,
    and each engine token must be the next one the decoder forced."""
    decoder = session.decoder()
    forced = []
    for index, token in enumerate(stream):
        token_id = token["id"]
        if token["emitter"] == "engine":
            assert forced and forced[0] == token_id, f"token {index}: the engine emits {token_id}, forced {forced}"
            forced.pop(0)
            continue
        assert not forced, f"token {index}: the engine forces {forced} before {token_id}"
        assert mask_admits(decoder.admissible(), token_id), f"token {index}: {session.vocabulary.name(token_id)} is outside the mask"
        forced, _ = decoder.step(token_id)
        forced = list(forced)
    assert not forced, f"the stream ends with {forced} forced"
    assert decoder.ended, "the decode has not ended at a consistent relation"
    return decoder


start = time.time()
decoder = decode(session, stream)
attempts = decoder.attempts()
print("decoded in", round(time.time() - start, 2), "s")
assert attempts is not None and len(attempts) == task.test_count
assert all(1 <= len(per_input) <= 2 for per_input in attempts), attempts

In [ ]:
with open("attempts.json", "w") as f:
    json.dump(attempts, f)
for index, per_input in enumerate(attempts):
    print("test input", index, "attempts:", len(per_input))
    for grid in per_input[:1]:
        print("\n".join(" ".join(str(cell) for cell in row) for row in grid))